# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TEJAS7356/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### My baseline rule

Prioritize content that has not been updated recently and has sufficient search visibility. Assign a higher score to content that meets both conditions so it can be reviewed first. This is a simple, transparent rule-based baseline for comparison with the Week 5 ML model.

### Reason code

- `STALE_BUT_VISIBLE` — the content is old and has enough search visibility to justify reviewing it.

### Action label

- `REVIEW_FOR_REFRESH` — review the content and decide whether an update is needed.

In [12]:
import os
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")

con.execute(
    f"CREATE OR REPLACE SECRET hf "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

print("DuckDB connected successfully!")

DuckDB connected successfully!


In [13]:
REL = "hf://datasets/FlyRank/internship-warehouse"

FACT = f"{REL}/fact_content_daily_performance"
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

print("February and March data paths are ready!")

February and March data paths are ready!


In [14]:
# Signal 1: Search visibility bucket check
signal1 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_impressions < 100 THEN 'Low: <100'
            WHEN gsc_impressions < 500 THEN 'Medium: 100-499'
            ELSE 'High: 500+'
        END AS impressions_bucket,
        COUNT(*) AS n,
        ROUND(AVG(gsc_clicks), 2) AS avg_clicks
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions IS NOT NULL
    GROUP BY impressions_bucket
    ORDER BY MIN(
        CASE
            WHEN gsc_impressions < 100 THEN 1
            WHEN gsc_impressions < 500 THEN 2
            ELSE 3
        END
    )
""").df()

display(signal1)

,impressions_bucket,n,avg_clicks
0,Low: <100,2192382,0.06
1,Medium: 100-499,369914,0.68
2,High: 500+,59487,3.49


In [15]:
# Signal 2: CTR versus average search position
signal2 = con.sql(f"""
    SELECT
        CASE
            WHEN gsc_avg_position < 10 THEN 'Top 10'
            WHEN gsc_avg_position < 20 THEN 'Positions 10-19'
            ELSE 'Position 20+'
        END AS position_bucket,
        COUNT(*) AS n,
        ROUND(AVG(
            100.0 * gsc_clicks / NULLIF(gsc_impressions, 0)
        ), 3) AS avg_ctr_pct
    FROM {FEB}
    WHERE gsc_data_available IS TRUE
      AND gsc_impressions > 0
      AND gsc_avg_position > 0
    GROUP BY position_bucket
    ORDER BY MIN(
        CASE
            WHEN gsc_avg_position < 10 THEN 1
            WHEN gsc_avg_position < 20 THEN 2
            ELSE 3
        END
    )
""").df()

display(signal2)

,position_bucket,n,avg_ctr_pct
0,Top 10,1550137,0.385
1,Positions 10-19,444504,0.247
2,Position 20+,447188,0.159


### Signal check verdicts

1. **Search visibility — CONFIRMED:** Rows with more impressions have higher average clicks. The average rises from 0.06 in the low-impression bucket to 3.49 in the high-impression bucket.
2. **Search position versus CTR — CONFIRMED:** Average CTR decreases from 0.385% for the Top 10 bucket to 0.159% for positions 20 and above.

### Baseline rule

Prioritize content with high search visibility for review when its search position is relatively poor. The rule will use impressions and average search position to identify content that has visibility but may have room for improvement.

### Reason code

- `VISIBLE_LOW_POSITION` — content has sufficient impressions but an average search position outside the Top 10.

### Action label

- `REVIEW_SEARCH_PERFORMANCE` — review the content's search performance to decide whether improvements are warranted.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 2: Build the baseline ranked queue

import os

# Aggregate daily records to one row per client and content item
queue = con.sql(f"""
    WITH content_summary AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS impressions,
            SUM(gsc_clicks) AS clicks,
            SUM(gsc_sum_position) * 1.0
                / NULLIF(SUM(gsc_impressions), 0) AS avg_position
        FROM {FEB}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    ),
    scored AS (
        SELECT
            *,
            CASE
                WHEN impressions >= 500
                 AND avg_position >= 10
                THEN impressions
                ELSE 0
            END AS score
        FROM content_summary
    )
    SELECT
        client_hash_id,
        content_hash_id,
        impressions,
        clicks,
        avg_position,
        score,
        CASE
            WHEN score > 0 THEN 'VISIBLE_LOW_POSITION'
            ELSE NULL
        END AS reason_code,
        CASE
            WHEN score > 0 THEN 'REVIEW_SEARCH_PERFORMANCE'
            ELSE 'NO_ACTION'
        END AS action_label
    FROM scored
    ORDER BY score DESC, impressions DESC
""").df()

# Add ranking
queue.insert(0, "rank", range(1, len(queue) + 1))

# Write the required CSV
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue.to_csv(output_path, index=False)

print("Ranked queue created successfully!")
print("Total content items:", len(queue))
print("Items flagged for review:", (queue["score"] > 0).sum())
print("Saved to:", output_path)

display(queue.head(10))

Ranked queue created successfully!
Total content items: 153559
Items flagged for review: 12795
Saved to: work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,impressions,clicks,avg_position,score,reason_code,action_label
0,1,client_23a62021009f63c4,content_e8a52cf3d5988c07,162129.0,627.0,13.584411,162129.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE
1,2,client_23a62021009f63c4,content_36e53e9c707674fc,100736.0,237.0,34.405029,100736.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE
2,3,client_23a62021009f63c4,content_df47d1b976106de4,85935.0,103.0,17.186094,85935.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE
3,4,client_fef1a8f436438636,content_84a6bf3578312e90,79986.0,74.0,19.783062,79986.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE
4,5,client_23a62021009f63c4,content_5e1c049f62e33b11,73072.0,118.0,16.379119,73072.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE
5,6,client_fef1a8f436438636,content_ba462518dad435fc,69134.0,39.0,27.477479,69134.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE
6,7,client_23a62021009f63c4,content_3df3f32f3fd58dea,68216.0,155.0,24.829497,68216.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE
7,8,client_23a62021009f63c4,content_b51957d7f4abe47e,57558.0,36.0,28.411064,57558.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE
8,9,client_23a62021009f63c4,content_bdf60c86117079be,51346.0,8.0,33.837300,51346.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE
9,10,client_fef1a8f436438636,content_0aaa197051f58d6f,49903.0,35.0,34.489710,49903.0,VISIBLE_LOW_POSITION,REVIEW_SEARCH_PERFORMANCE


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [17]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Section 3: Top-10 review
top10 = queue.head(10).copy()

top10["why_ranked_here"] = (
    "At least 500 impressions and average position >= 10; "
    "ranked by impressions."
)

top10["what_could_make_it_wrong"] = (
    "The position may not reflect the same opportunity for every query; "
    "the content may already be relevant, or the data may be incomplete."
)

display(
    top10[
        [
            "rank",
            "impressions",
            "clicks",
            "avg_position",
            "action_label",
            "reason_code",
            "why_ranked_here",
            "what_could_make_it_wrong",
        ]
    ]
)

,rank,impressions,clicks,avg_position,action_label,reason_code,why_ranked_here,what_could_make_it_wrong
0,1,162129.0,627.0,13.584411,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...
1,2,100736.0,237.0,34.405029,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...
2,3,85935.0,103.0,17.186094,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...
3,4,79986.0,74.0,19.783062,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...
4,5,73072.0,118.0,16.379119,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...
5,6,69134.0,39.0,27.477479,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...
6,7,68216.0,155.0,24.829497,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...
7,8,57558.0,36.0,28.411064,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...
8,9,51346.0,8.0,33.837300,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...
9,10,49903.0,35.0,34.489710,REVIEW_SEARCH_PERFORMANCE,VISIBLE_LOW_POSITION,At least 500 impressions and average position ...,The position may not reflect the same opportun...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Weak picks

- **Weak pick 1:** A page with many impressions but a very poor average position. High visibility alone does not prove that updating the content will improve performance.
- **Weak pick 2:** A page with enough impressions and a position just outside the Top 10. Its ranking may not represent a realistic improvement opportunity.
- **Weak pick 3:** A page with low clicks despite high impressions. The cause could be query intent, the search snippet, or other factors rather than outdated content.

### Leakage check

The baseline uses February 2026 search impressions, clicks, and average search position. It does not use a future-month label or a product-generated recommendation flag. The rule is hand-written and uses fixed thresholds.

The current queue is based on February data only. Before claiming the leakage check is fully verified, confirm that no future-window fields or product flags are used in the scoring code.

In [18]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.